# Lab4 — Rule-based IE (corrected)

Це виправлена версія ноутбука з кращим LOCATION (відмінкові форми міст).

## 1) Install deps

In [1]:
!pip -q install -r ../requirements.txt

## 2) Load data from Lab2

In [2]:
from pathlib import Path
import pandas as pd
import sys, json, importlib

LAB4_ROOT = Path("..").resolve()
LAB2_ROOT = (LAB4_ROOT.parent / "project_lab2").resolve()
sys.path.insert(0, str(LAB4_ROOT))

v2_path = LAB2_ROOT / "data" / "processed_v2" / "processed_v2.csv"
print("Reading:", v2_path)

df = pd.read_csv(v2_path)
print("Shape:", df.shape)
df.head()

Reading: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\processed_v2\processed_v2.csv
Shape: (1000, 4)


,text_id,text,sentences,label
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...","[""Вступив на ІСТ цього року, тепер молюся, щоб...",Question / Request for Help
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[""Цифрова держава Повідомлення 123 від 18.04.2...",Question / Request for Help
2,3099,Старий університет поки що вчить. Наразі налаш...,"[""Старий університет поки що вчить."", ""Наразі ...",Neutral Comment
3,8664,"На пл. Ринок ЦНАП м.Львова, швидке ьа якісне в...","[""На пл. Ринок ЦНАП м.Львова, швидке ьа якісне...",Gratitude / Positive Feedback
4,1035,"Мені здається, що наша кузня супер-кадрів в IT...","[""Мені здається, що наша кузня супер-кадрів в ...",Suggestion / Idea


## 3) Reload corrected rules

In [3]:
import src.ie_rules
import importlib
importlib.reload(src.ie_rules)

from src.ie_rules import extract_dates, extract_locations, extract_doc_ids, extract_all

## 4) Run extraction on real sample

In [4]:
sample = df.sample(15, random_state=42).copy()
sample["ie"] = sample["text"].astype(str).apply(extract_all)
sample[["text_id", "text", "ie"]]

,text_id,text,ie
521,7043,Операторів 12. А фотографів 6. З них пів дня п...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
737,9294,"Чернігівська область, м. Мена, вул. Сіверський...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
740,3825,"Напевно, був би кращім місцем, якби не було ко...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
660,925,"Щоб там щодня, крім вихідних. Нічого особливог...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
411,10001377,Тільки в епіцентрі працює? А в леруа мерлен на...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
678,1600,Реєстрація ФОП покроково з врахуванням нововве...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
626,16151,Навчальний заклад розташований в найголовнішій...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
513,1871,Це незвичайний навчальний заклад. Якщо шукаєте...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
859,4358,Харківський університет називають каразінським...,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
136,16189,"Як в живу, як на фото, дуже гарно виглядає, сю...","{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"


## 5) Build weak-gold seed

In [5]:
cand = df[['text_id', 'text', 'label']].copy()

# broad candidate patterns for weak-gold seed mining
date_pat = r"\b\d{1,2}[./-]\d{1,2}[./-]\d{4}\b|\b\d{1,2}\s+(січня|лютого|березня|квітня|травня|червня|липня|серпня|вересня|жовтня|листопада|грудня)\b"
city_pat = r"\b(Київ|Львів|Харків|Одеса|Дніпро|Чернівці|Суми|Полтава|Тернопіль|Івано-Франківськ|Ужгород|Запоріжжя|Миколаїв|Херсон|Черкаси|Житомир|Рівне|Луцьк|Хмельницький|Вінниця|Чернігів|Кропивницький)\w*\b"
doc_pat = r"\b(повідомлення|заява|документ|звернення|рішення|договір|запит|лист)\s*№?\s*\d{1,10}\b|№\s*\d{1,10}\b"

date_mask = cand['text'].astype(str).str.contains(date_pat, regex=True, case=False, na=False)
city_mask = cand['text'].astype(str).str.contains(city_pat, regex=True, case=False, na=False)
doc_mask = cand['text'].astype(str).str.contains(doc_pat, regex=True, case=False, na=False)

date_seed = cand.loc[date_mask, ['text_id', 'text', 'label']].head(15)
city_seed = cand.loc[city_mask, ['text_id', 'text', 'label']].head(15)
doc_seed = cand.loc[doc_mask, ['text_id', 'text', 'label']].head(15)

gold_seed = pd.concat([date_seed, city_seed, doc_seed], ignore_index=True).drop_duplicates(subset=['text_id'])
gold_seed = gold_seed.reset_index(drop=True)

print('Selected texts for weak-gold subset:', gold_seed.shape)
gold_seed.head(20)


Selected texts for weak-gold subset: (9, 3)


C:\Users\maia1\AppData\Local\Temp\ipykernel_43496\1717636250.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  date_mask = cand['text'].astype(str).str.contains(date_pat, regex=True, case=False, na=False)
C:\Users\maia1\AppData\Local\Temp\ipykernel_43496\1717636250.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  city_mask = cand['text'].astype(str).str.contains(city_pat, regex=True, case=False, na=False)
C:\Users\maia1\AppData\Local\Temp\ipykernel_43496\1717636250.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  doc_mask = cand['text'].astype(str).str.contains(doc_pat, regex=True, case=False, na=False)


,text_id,text,label
0,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,Question / Request for Help
1,3228,"Був у центрі 15.11.2021року Просто супер, жінк...",Gratitude / Positive Feedback
2,1612,"податкова основянського району, години роботи ...",Neutral Comment
3,7010,Чи не сподобалося. Витрачений цілий день. А як...,Complaint / Dissatisfaction
4,8889,Жахливо! Прийшла 18.01.2022...вистояла чергу б...,Complaint / Dissatisfaction
5,9365,31.05.2025 здавала практичний іспит в даному Т...,Gratitude / Positive Feedback
6,1572,Інформація станом на 14.07.2023 Індустріальна ...,Neutral Comment
7,10000441,"Так,інформація буде надана 14.07.2023р.о 14-00...",Question / Request for Help
8,10004160,Доброго вечора! <URL> Василь Матійчук родом із...,Question / Request for Help


## 6) Build weak-gold

In [6]:
gold_records = []

for _, row in gold_seed.iterrows():
    text_id = int(row["text_id"])
    text = str(row["text"])
    auto_gold = extract_all(text)

    for field_type, items in auto_gold.items():
        for item in items:
            gold_records.append({
                "text_id": text_id,
                "text": text,
                "field_type": field_type,
                "span_text": item.get("raw_value", item["value"]),
                "start_char": item["start_char"],
                "end_char": item["end_char"],
                "normalized_value": item["value"],
            })

gold_df = pd.DataFrame(gold_records)
print("Weak-gold rows:", gold_df.shape)
gold_df.head(20)

Weak-gold rows: (9, 7)


,text_id,text,field_type,span_text,start_char,end_char,normalized_value
0,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,DATE,18.04.2023,37,47,2023-04-18
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,DOC_ID,Повідомлення 123,16,32,123
2,1612,"податкова основянського району, години роботи ...",DATE,26.03.2019,59,69,2019-03-26
3,1612,"податкова основянського району, години роботи ...",DOC_ID,№ 7,129,132,7
4,7010,Чи не сподобалося. Витрачений цілий день. А як...,DATE,10.07.2017,106,116,2017-07-10
5,8889,Жахливо! Прийшла 18.01.2022...вистояла чергу б...,DATE,18.01.2022,17,27,2022-01-18
6,9365,31.05.2025 здавала практичний іспит в даному Т...,DATE,31.05.2025,0,10,2025-05-31
7,1572,Інформація станом на 14.07.2023 Індустріальна ...,DATE,14.07.2023,21,31,2023-07-14
8,10004160,Доброго вечора! <URL> Василь Матійчук родом із...,DATE,08.04.2023,210,220,2023-04-08


## 7) Save weak-gold subset

In [7]:
gold_path = LAB4_ROOT / "data" / "sample" / "lab4_gold_ie.jsonl"

with gold_path.open("w", encoding="utf-8") as f:
    for _, row in gold_df.iterrows():
        rec = {
            "text_id": int(row["text_id"]),
            "text": str(row["text"]),
            "field_type": str(row["field_type"]),
            "span_text": str(row["span_text"]),
            "start_char": int(row["start_char"]),
            "end_char": int(row["end_char"]),
            "normalized_value": str(row["normalized_value"]),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved weak-gold subset:", gold_path)

Saved weak-gold subset: C:\Users\maia1\data\politiekh\masters\nlp\project_lab4\data\sample\lab4_gold_ie.jsonl


## 8) Evaluate precision

In [8]:
from collections import defaultdict

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

edge_rows = load_jsonl(LAB4_ROOT / 'tests' / 'ie_edge_cases.jsonl')

stats = {
    'DATE': {'tp': 0, 'fp': 0, 'fn': 0, 'skipped': 0},
    'LOCATION': {'tp': 0, 'fp': 0, 'fn': 0, 'skipped': 0},
    'DOC_ID': {'tp': 0, 'fp': 0, 'fn': 0, 'skipped': 0},
}
fp_examples = []

for r in edge_rows:
    ft = r['field_type']
    text = str(r['raw_text'])
    expected = str(r['expected_behavior']).lower()
    pred_items = extract_all(text).get(ft, [])
    pred_positive = len(pred_items) > 0

    # ambiguous cases are tracked but excluded from strict precision
    if ('можна пропустити' in expected) or ('перевірити можливий конфлікт' in expected):
        stats[ft]['skipped'] += 1
        continue

    expect_positive = 'витягнути' in expected
    expect_negative = ('не вважати' in expected) or ('не витягувати' in expected)

    if expect_positive:
        if pred_positive:
            stats[ft]['tp'] += 1
        else:
            stats[ft]['fn'] += 1
    elif expect_negative:
        if pred_positive:
            stats[ft]['fp'] += 1
            fp_examples.append({
                'id': r['id'],
                'field_type': ft,
                'text': text,
                'predicted_value': pred_items[0]['value'] if pred_items else None,
                'expected_behavior': r['expected_behavior'],
            })

precision_rows = []
for ft in ['DATE', 'LOCATION', 'DOC_ID']:
    tp = stats[ft]['tp']
    fp = stats[ft]['fp']
    fn = stats[ft]['fn']
    pred_n = tp + fp
    precision = tp / pred_n if pred_n else None
    recall = tp / (tp + fn) if (tp + fn) else None
    precision_rows.append({
        'field_type': ft,
        'predicted': pred_n,
        'correct': tp,
        'precision': precision,
        'recall': recall,
        'skipped_ambiguous': stats[ft]['skipped'],
    })

precision_df = pd.DataFrame(precision_rows)
precision_df


,field_type,predicted,correct,precision,recall,skipped_ambiguous
0,DATE,4,4,1.0,1.0,0
1,LOCATION,5,5,1.0,1.0,1
2,DOC_ID,7,7,1.0,1.0,1


## 9) False positives / problem cases

In [9]:
fp_df = pd.DataFrame(fp_examples)
print("False positives found:", len(fp_df))
fp_df.head(10)

False positives found: 0


""


## 10) Edge cases

In [10]:
edge_rows = load_jsonl(LAB4_ROOT / "tests" / "ie_edge_cases.jsonl")
edge_df = pd.DataFrame(edge_rows)
edge_df["prediction"] = edge_df["raw_text"].apply(extract_all)
edge_df[["id", "field_type", "expected_behavior", "raw_text", "prediction"]].head(25)

,id,field_type,expected_behavior,raw_text,prediction
0,ie01,DOC_ID,витягнути DOC_ID=123,Повідомлення 123 від 18.04.2024,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
1,ie02,DATE,витягнути DATE=2024-04-18,Повідомлення 123 від 18.04.2024,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
2,ie03,DOC_ID,витягнути DOC_ID=45,Заява №45 подана 5 травня 2023,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
3,ie04,DATE,витягнути DATE=2023-05-05,Заява №45 подана 5 травня 2023,"{'DATE': [{'field_type': 'DATE', 'value': '202..."
4,ie05,DATE,не вважати 1.2.3 датою,Версія 1.2.3 не працює,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
5,ie06,DATE,не вважати 3.14 датою,Число 3.14 не є датою,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
6,ie07,LOCATION,витягнути LOCATION=Львів,Місто Львів дуже гарне,"{'DATE': [], 'LOCATION': [{'field_type': 'LOCA..."
7,ie08,LOCATION,витягнути LOCATION=Харків,м. Харків працює стабільно,"{'DATE': [], 'LOCATION': [{'field_type': 'LOCA..."
8,ie09,LOCATION,не витягувати прикметник як місто,Львівська область,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
9,ie10,DOC_ID,витягнути DOC_ID=77,Документ №77 зареєстровано,"{'DATE': [], 'LOCATION': [], 'DOC_ID': [{'fiel..."


## 11) Focused problem cases

In [11]:
problem_cases = edge_df[
    edge_df["raw_text"].str.contains("Суми|№2024|1.2.3|3.14|Львівська|Києві|Львові", regex=True, na=False)
].copy()

problem_cases[["id", "field_type", "expected_behavior", "raw_text", "prediction"]]

,id,field_type,expected_behavior,raw_text,prediction
4,ie05,DATE,не вважати 1.2.3 датою,Версія 1.2.3 не працює,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
5,ie06,DATE,не вважати 3.14 датою,Число 3.14 не є датою,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
8,ie09,LOCATION,не витягувати прикметник як місто,Львівська область,"{'DATE': [], 'LOCATION': [], 'DOC_ID': []}"
14,ie15,LOCATION,перевірити можливий конфлікт з містом Суми,Суми зросли на 10%,"{'DATE': [], 'LOCATION': [{'field_type': 'LOCA..."
24,ie25,DOC_ID,"витягнути DATE, LOCATION, DOC_ID",5 липня 2021 у Львові подали заяву №9,"{'DATE': [{'field_type': 'DATE', 'value': '202..."


## 12) Save audit summary

In [12]:
doc = LAB4_ROOT / 'docs' / 'audit_summary_lab4.md'

lines = []
lines.append('# Audit summary — Lab4')
lines.append('')
lines.append('## Precision table (edge-case based)')
lines.append('')

for _, r in precision_df.iterrows():
    p = 'N/A' if pd.isna(r['precision']) else f"{r['precision']:.4f}"
    rec = 'N/A' if pd.isna(r['recall']) else f"{r['recall']:.4f}"
    lines.append(
        f"- {r['field_type']}: predicted={int(r['predicted'])}, correct={int(r['correct'])}, "
        f"precision={p}, recall={rec}, skipped_ambiguous={int(r['skipped_ambiguous'])}"
    )

lines.append('')
lines.append('## Notes')
lines.append('')
lines.append('Оцінка зроблена на ie_edge_cases.jsonl (не на self-generated weak-gold).')
lines.append('Weak-gold збережено як допоміжний набір прикладів, але не як незалежний тест.')
lines.append('Precision-first anti-rules зменшують хибні DOC_ID/DATE спрацювання.')

doc.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('Saved:', doc)

# Update Lab4 README with real numbers
readme = LAB4_ROOT / 'labs' / 'lab04' / 'README.md'
rr = []
rr.append('# LPNU NLP — Lab 04 (Rule-based IE)')
rr.append('')
rr.append('## 1) Які 3 типи полів витягуються')
rr.append('- DATE')
rr.append('- LOCATION')
rr.append('- DOC_ID')
rr.append('')
rr.append('## 2) Які правила/словники використано')
rr.append('- regex для числових і текстових дат')
rr.append('- словник міст + відмінкові форми для LOCATION')
rr.append('- regex + контекстні ключові слова для DOC_ID')
rr.append('- anti-rules для 1.2.3 / 3.14 / year-like DOC_ID')
rr.append('')
rr.append('## 3) Precision (edge-case based)')
for _, r in precision_df.iterrows():
    p = 'N/A' if pd.isna(r['precision']) else f"{r['precision']:.4f}"
    rr.append(f"- {r['field_type']}: {p}")
rr.append('')
rr.append('## 4) Топ-5 edge cases')
rr.append('- дати vs версії (1.2.3) і десяткові числа (3.14)')
rr.append('- city exact match vs прикметники (Львівська)')
rr.append('- Суми як місто vs загальна лексема')
rr.append('- рікоподібні ID (№2024)')
rr.append('- числа без контексту не повинні ставати DOC_ID')
rr.append('')
rr.append('## 5) Що планується покращити')
rr.append('- розширити словники міст/контекстів')
rr.append('- додати більше anti-rules після аналізу false positives')
rr.append('- вручну розмітити більший gold subset (30–50 текстів)')
readme.write_text('\n'.join(rr) + '\n', encoding='utf-8')
print('Saved:', readme)

# Update dataset card placeholder
dc = LAB4_ROOT / 'docs' / 'dataset_card.md'
dc_lines = []
dc_lines.append('# Dataset Card — Lab4 update')
dc_lines.append('')
dc_lines.append('## IE fields')
dc_lines.append('- DATE')
dc_lines.append('- LOCATION')
dc_lines.append('- DOC_ID')
dc_lines.append('')
dc_lines.append('## Privacy & masking')
dc_lines.append('- Inputs come from processed_v2 with URL/EMAIL/PHONE masking from Lab2.')
dc_lines.append('- Rule extraction works on masked text to reduce direct PII exposure.')
dc_lines.append('')
dc_lines.append('## Precision-first policy')
dc_lines.append('- Conservative anti-rules reduce false positives for DATE/DOC_ID.')
dc_lines.append('- Ambiguous cases are tracked separately and excluded from strict precision.')
dc_lines.append('')
dc_lines.append('## Current metrics (edge cases)')
for _, r in precision_df.iterrows():
    p = 'N/A' if pd.isna(r['precision']) else f"{r['precision']:.4f}"
    dc_lines.append(f"- {r['field_type']} precision: {p}")
dc.write_text('\n'.join(dc_lines) + '\n', encoding='utf-8')
print('Saved:', dc)


Saved:

 C:\Users\maia1\data\politiekh\masters\nlp\project_lab4\docs\audit_summary_lab4.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab4\labs\lab04\README.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab4\docs\dataset_card.md
